In [ ]:
#!pip install spacy

In [1]:
"""
Extracción de Entidades Nombradas (NER) — Documentos 23F
=========================================================
Detecta y cuenta automáticamente:
  - PER  : personas (Tejero, Armada, Milans del Bosch...)
  - LOC  : lugares (Madrid, Congreso, Valencia...)
  - ORG  : organizaciones (Guardia Civil, CESID, Gobierno...)
  - MISC : otros (fechas, eventos, operaciones...)

Genera:
  - personas.csv / lugares.csv / organizaciones.csv
  - ner_barras.png  — top entidades por categoría
  - ner_documentos.png — en cuántos documentos aparece cada entidad

Requisitos:
    pip install spacy matplotlib pandas
    python -m spacy download es_core_news_lg

Uso:
    python ner_23f.py
"""

import os
import csv
import re
from pathlib import Path
from collections import defaultdict, Counter
from nltk.corpus import stopwords

import spacy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


# ══════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════
CARPETA_DOCS = r"C:\Javier Pardo\Machine Learning\23F_Javier_Pardo\documentos_23f"
OUTPUT_DIR   = r"C:\Javier Pardo\Machine Learning\23F_Javier_Pardo\analisis_23f"
TOP_N        = 50   # entidades a mostrar por categoría en los gráficos

# Entidades a ignorar (falsos positivos comunes — ajusta según resultados)
IGNORAR = {
    # Palabras genéricas que spaCy confunde con entidades
    "señor", "señores", 
    #"general", "coronel", "teniente",
    #"presidente", "ministro", "página", "documento",
    # Metadatos del scraper
    "ocr", "mistral", "rtve", "resumen", "título",
}
# ══════════════════════════════════════════════════════



In [2]:
def leer_documentos(carpeta):
    """Lee todos los .txt y devuelve lista de (nombre_archivo, texto_transcripcion)."""
    docs = []
    archivos = sorted(Path(carpeta).glob("*.txt"))
    archivos = [a for a in archivos if not a.name.startswith("_")]

    for archivo in archivos:
        with open(archivo, "r", encoding="utf-8") as f:
            contenido = f.read()
        # Quedarnos solo con la transcripción
        if "═" * 10 in contenido:
            texto = contenido.split("═" * 10)[-1].strip()
        else:
            texto = contenido.strip()
        if texto:
            docs.append((archivo.stem, texto))

    print(f"  → {len(docs)} documentos leídos")
    return docs



In [39]:
def limpiar_texto(texto):
    """
    1. Minúsculas
    2. Elimina números y puntuación
    3. Divide en palabras
    4. Elimina stopwords y palabras cortas
    """
    
    # Cargar stopwords en español
    stops = set(stopwords.words("spanish"))
    #stops.update(STOPWORDS_EXTRA)
    #stops={}
    
    # Minúsculas
    texto = [(a, b.lower()) for a, b in texto]

    # Eliminar números y caracteres especiales, conservar letras y espacios
    #texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    #texto = [(a, b.lower()) for a, b in texto]
    #re.sub(r"[^a-záéíóúüñ\s]", " ", docs[0][1])
    '''
    for a in range(len(texto)):
        texto[a][1] = re.sub(r"\s+", " ", re.sub(r"[^a-záéíóúüñ\s]", " ", texto[a][1].replace("\n", " "))).strip()
        texto[a][0] = (texto[a][0], texto)
    '''   
        
    for i, (a, b) in enumerate(texto):
        if isinstance(b, str):
    ### El error esta en que no uso en la segunda lo que hice en la primera, debo tener una variable temporal
    # primera        docs[i] = (a, b.lower())
    # segunda        docs[i] = (a, re.sub(r"\s+", " ", re.sub(r"[^a-záéíóúüñ\s]", " ", b.replace("\n", " "))).strip())
            clean = b.lower()
            #clean = re.sub(r"\s+", " ", re.sub(r"[^a-záéíóúüñ0-9\s]", " ", clean.replace("\n", " "))).strip()
            clean = clean.replace("\n", " ")
            clean = re.sub(r"[^a-záéíóúüñ0-9\s]", " ",clean)
            clean = re.sub(r"\s+", " ", clean).strip()
            
            clean = clean.split() # Dividir en palabras
                # Filtrar: sin stopwords, sin palabras cortas
            palabras_limpias = [
                p for p in clean
                if p not in stops and len(p) >= 2 # Minima longitud de las palabras
            ]            
            texto[i] = (a,palabras_limpias)

       
    return texto

<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
C:\Users\jhpar\AppData\Local\Temp\ipykernel_24700\495237656.py:23: SyntaxWarning: invalid escape sequence '\s'
  texto[a][1] = re.sub(r"\s+", " ", re.sub(r"[^a-záéíóúüñ\s]", " ", texto[a][1].replace("\n", " "))).strip()


In [4]:
def procesar_ner(docs, nlp):
    """
    Procesa cada documento con spaCy y extrae todas las entidades.
    Devuelve:
      - entidades_total : Counter {(texto, tipo): count}
      - entidades_docs  : dict    {(texto, tipo): set de documentos donde aparece}
      - entidades_x_doc : dict    {nombre_doc: lista de entidades}
    """
    entidades_total = Counter()
    entidades_docs  = defaultdict(set)
    entidades_x_doc = {}

    total = len(docs)
    for i, (nombre, texto) in enumerate(docs, 1):
        print(f"  [{i:>3}/{total}] {nombre[:50]}...", end="\r")

        # spaCy tiene límite de caracteres — procesamos en fragmentos si es muy largo
        MAX_CHARS = 100_000
        fragmentos = [texto[j:j+MAX_CHARS] for j in range(0, len(texto), MAX_CHARS)]

        entidades_doc = []
        for fragmento in fragmentos:
            doc_nlp = nlp(fragmento)
            for ent in doc_nlp.ents:
                texto_ent = ent.text.strip()
                tipo_ent  = ent.label_

                # Filtros de calidad
                if len(texto_ent) < 2:
                    continue
                if texto_ent.lower() in IGNORAR:
                    continue
                if tipo_ent not in ("PER", "LOC", "ORG", "MISC"):
                    continue
                # Ignorar entidades que son solo números
                if texto_ent.replace(".", "").replace(",", "").isdigit():
                    continue

                clave = (texto_ent, tipo_ent)
                entidades_total[clave] += 1
                entidades_docs[clave].add(nombre)
                entidades_doc.append(clave)

        entidades_x_doc[nombre] = entidades_doc

    print(f"\n  → {len(entidades_total)} entidades únicas encontradas")
    return entidades_total, entidades_docs, entidades_x_doc



In [5]:
def guardar_csvs(entidades_total, entidades_docs, output_dir):
    """Guarda un CSV por categoría con frecuencia y número de documentos."""
    categorias = {
        "PER":  "personas",
        "LOC":  "lugares",
        "ORG":  "organizaciones",
        "MISC": "miscelaneo",
    }

    resumen = {}
    for tipo, nombre_archivo in categorias.items():
        filas = [
            {
                "entidad":     texto,
                "tipo":        tipo,
                "menciones":   entidades_total[(texto, tipo)],
                "documentos":  len(entidades_docs[(texto, tipo)]),
            }
            for (texto, t) in entidades_total
            if t == tipo
        ]
        filas.sort(key=lambda x: x["menciones"], reverse=True)

        ruta = os.path.join(output_dir, f"{nombre_archivo}.csv")
        with open(ruta, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["entidad", "tipo", "menciones", "documentos"])
            writer.writeheader()
            writer.writerows(filas)

        resumen[tipo] = filas
        print(f"  ✔ {ruta}  ({len(filas)} entidades)")

    return resumen



In [6]:
def grafico_barras_ner(resumen, top_n, output_dir):
    """
    Gráfico con 4 subgráficos (uno por categoría),
    mostrando las top N entidades por número de menciones.
    """
    categorias = [
        ("PER",  "Personas",        "#7F77DD"),
        ("LOC",  "Lugares",         "#1D9E75"),
        ("ORG",  "Organizaciones",  "#BA7517"),
        ("MISC", "Miscelánea",      "#888780"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    fig.suptitle(
        "Entidades nombradas más frecuentes\nDocumentos desclasificados 23F",
        fontsize=14, fontweight="bold", y=1.01
    )
    axes = axes.flatten()

    for ax, (tipo, titulo, color) in zip(axes, categorias):
        filas = resumen.get(tipo, [])[:top_n]
        if not filas:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center",
                    transform=ax.transAxes, color="#aaa")
            ax.set_title(titulo)
            continue

        nombres = [f["entidad"] for f in filas]
        valores = [f["menciones"] for f in filas]
        max_v   = max(valores)
        colores = [plt.matplotlib.colors.to_rgba(color, alpha=0.4 + 0.6 * v / max_v)
                   for v in valores]

        barras = ax.barh(nombres[::-1], valores[::-1], color=colores[::-1])

        for barra, val in zip(barras, valores[::-1]):
            ax.text(
                barra.get_width() + max_v * 0.01,
                barra.get_y() + barra.get_height() / 2,
                str(val), va="center", ha="left", fontsize=8, color="#555"
            )

        ax.set_title(titulo, fontsize=12, fontweight="bold", color=color)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="y", labelsize=9)
        ax.set_xlabel("Menciones", fontsize=9)

    plt.tight_layout()
    ruta = os.path.join(output_dir, "ner_barras.png")
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✔ {ruta}")



In [7]:
def grafico_documentos(resumen, top_n, output_dir):
    """
    Gráfico de burbujas: eje X = menciones totales,
    eje Y = número de documentos distintos donde aparece.
    Muestra personas y organizaciones (los más interesantes).
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(
        "Presencia en documentos vs. menciones totales\n(cada burbuja = una entidad)",
        fontsize=13, fontweight="bold"
    )

    configs = [
        ("PER",  "Personas",       "#7F77DD", axes[0]),
        ("ORG",  "Organizaciones", "#BA7517", axes[1]),
    ]

    for tipo, titulo, color, ax in configs:
        filas = resumen.get(tipo, [])[:top_n * 2]
        if not filas:
            continue

        x = [f["menciones"]  for f in filas]
        y = [f["documentos"] for f in filas]
        nombres = [f["entidad"] for f in filas]

        ax.scatter(x, y, s=[v * 3 for v in x], color=color, alpha=0.5, edgecolors="white")

        # Etiquetar los más destacados
        max_m = max(x)
        for xi, yi, nombre in zip(x, y, nombres):
            if xi > max_m * 0.3 or yi > max(y) * 0.5:
                ax.annotate(
                    nombre,
                    (xi, yi),
                    textcoords="offset points",
                    xytext=(5, 5),
                    fontsize=8,
                    color="#333",
                )

        ax.set_xlabel("Menciones totales", fontsize=10)
        ax.set_ylabel("Número de documentos", fontsize=10)
        ax.set_title(titulo, fontsize=12, fontweight="bold", color=color)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()
    ruta = os.path.join(output_dir, "ner_documentos.png")
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✔ {ruta}")


In [8]:
def imprimir_resumen(resumen):
    """Muestra un resumen en la terminal."""
    etiquetas = {"PER": "PERSONAS", "LOC": "LUGARES", "ORG": "ORGANIZACIONES", "MISC": "MISC"}
    for tipo, titulo in etiquetas.items():
        filas = resumen.get(tipo, [])[:10]
        if not filas:
            continue
        print(f"\n  TOP 10 {titulo}:")
        for i, f in enumerate(filas, 1):
            print(f"    {i:>2}. {f['entidad']:<30} {f['menciones']:>4} menciones · {f['documentos']} docs")


In [ ]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"\n{'═'*58}")
    print(f"  NER — Extracción de entidades nombradas 23F")
    print(f"{'═'*58}\n")

    # 1. Cargar modelo spaCy
    print("🧠 Cargando modelo de español (es_core_news_lg)...")
    print("   (si es la primera vez puede tardar unos segundos)\n")
    nlp = spacy.load("es_core_news_lg")

    # 2. Leer documentos
    print("📂 Leyendo documentos...")
    docs = leer_documentos(CARPETA_DOCS)

    # 3. Procesar NER
    print(f"\n🔍 Analizando entidades en {len(docs)} documentos...")
    print("   (esto tarda ~2-3 minutos)\n")
    entidades_total, entidades_docs, entidades_x_doc = procesar_ner(docs, nlp)

    # 4. Guardar CSVs
    print("\n💾 Guardando CSVs...")
    resumen = guardar_csvs(entidades_total, entidades_docs, OUTPUT_DIR)

    # 5. Resumen en terminal
    imprimir_resumen(resumen)

    # 6. Gráfico de barras
    print("\n📊 Generando gráfico de barras...")
    grafico_barras_ner(resumen, TOP_N, OUTPUT_DIR)

    # 7. Gráfico de burbujas
    print("\n🫧  Generando gráfico de presencia en documentos...")
    grafico_documentos(resumen, TOP_N, OUTPUT_DIR)

    print(f"\n{'═'*58}")
    print(f"  ✅ NER completado")
    print(f"  📁 Resultados en: ./{OUTPUT_DIR}/")
    print(f"     - personas.csv")
    print(f"     - lugares.csv")
    print(f"     - organizaciones.csv")
    print(f"     - miscelaneo.csv")
    print(f"     - ner_barras.png")
    print(f"     - ner_documentos.png")
    print(f"{'═'*58}\n")
    print("  💡 CONSEJO: revisa los CSVs y añade a IGNORAR las")
    print("     entidades que sean falsos positivos. Luego vuelve")
    print("     a ejecutar para obtener resultados más limpios.\n")


if __name__ == "__main__":
    main()

In [ ]:
print(f"\n{'═'*58}")
print(f"  NER — Extracción de entidades nombradas 23F")
print(f"{'═'*58}\n")

# 1. Cargar modelo spaCy
print("🧠 Cargando modelo de español (es_core_news_lg)...")
print("   (si es la primera vez puede tardar unos segundos)\n")
nlp = spacy.load("es_core_news_lg")

# 2. Leer documentos
print("📂 Leyendo documentos...")
docs = leer_documentos(CARPETA_DOCS)


'''
# 3. Procesar NER
print(f"\n🔍 Analizando entidades en {len(palabras)} documentos...")
print("   (esto tarda ~2-3 minutos)\n")
entidades_total, entidades_docs, entidades_x_doc = procesar_ner(palabras, nlp)
'''
# 3. Procesar NER
#print(f"\n🔍 Analizando entidades en {len(docs)} documentos...")
#print("   (esto tarda ~2-3 minutos)\n")
#entidades_total, entidades_docs, entidades_x_doc = procesar_ner(docs, nlp)
'''

# 4. Guardar CSVs
print("\n💾 Guardando CSVs...")
resumen = guardar_csvs(entidades_total, entidades_docs, OUTPUT_DIR)

# 5. Resumen en terminal
imprimir_resumen(resumen)

# 6. Gráfico de barras
print("\n📊 Generando gráfico de barras...")
grafico_barras_ner(resumen, TOP_N, OUTPUT_DIR)

# 7. Gráfico de burbujas
print("\n🫧  Generando gráfico de presencia en documentos...")
grafico_documentos(resumen, TOP_N, OUTPUT_DIR)

print(f"\n{'═'*58}")
print(f"  ✅ NER completado")
print(f"  📁 Resultados en: ./{OUTPUT_DIR}/")
print(f"     - personas.csv")
print(f"     - lugares.csv")
print(f"     - organizaciones.csv")
print(f"     - miscelaneo.csv")
print(f"     - ner_barras.png")
print(f"     - ner_documentos.png")
print(f"{'═'*58}\n")
print("  💡 CONSEJO: revisa los CSVs y añade a IGNORAR las")
print("     entidades que sean falsos positivos. Luego vuelve")
print("     a ejecutar para obtener resultados más limpios.\n")


'''



══════════════════════════════════════════════════════════
  NER — Extracción de entidades nombradas 23F
══════════════════════════════════════════════════════════

🧠 Cargando modelo de español (es_core_news_lg)...
   (si es la primera vez puede tardar unos segundos)

📂 Leyendo documentos...
  → 157 documentos leídos

🧹 Limpiando texto...
  → 157 palabras tras limpiar
   (esto tarda ~2-3 minutos)

  [157/157] Índices_de_subversión_en_las_FAS._Marca_SECRETO_(d...
  → 15054 entidades únicas encontradas


'\n\n# 4. Guardar CSVs\nprint("\n💾 Guardando CSVs...")\nresumen = guardar_csvs(entidades_total, entidades_docs, OUTPUT_DIR)\n\n# 5. Resumen en terminal\nimprimir_resumen(resumen)\n\n# 6. Gráfico de barras\nprint("\n📊 Generando gráfico de barras...")\ngrafico_barras_ner(resumen, TOP_N, OUTPUT_DIR)\n\n# 7. Gráfico de burbujas\nprint("\n🫧  Generando gráfico de presencia en documentos...")\ngrafico_documentos(resumen, TOP_N, OUTPUT_DIR)\n\nprint(f"\n{\'═\'*58}")\nprint(f"  ✅ NER completado")\nprint(f"  📁 Resultados en: ./{OUTPUT_DIR}/")\nprint(f"     - personas.csv")\nprint(f"     - lugares.csv")\nprint(f"     - organizaciones.csv")\nprint(f"     - miscelaneo.csv")\nprint(f"     - ner_barras.png")\nprint(f"     - ner_documentos.png")\nprint(f"{\'═\'*58}\n")\nprint("  💡 CONSEJO: revisa los CSVs y añade a IGNORAR las")\nprint("     entidades que sean falsos positivos. Luego vuelve")\nprint("     a ejecutar para obtener resultados más limpios.\n")\n\n\n'

In [41]:
print(f"\n{'═'*58}")
print(f"  NER — Extracción de entidades nombradas 23F")
print(f"{'═'*58}\n")

# 1. Cargar modelo spaCy
print("🧠 Cargando modelo de español (es_core_news_lg)...")
print("   (si es la primera vez puede tardar unos segundos)\n")
nlp = spacy.load("es_core_news_lg")

# 2. Leer documentos
print("📂 Leyendo documentos...")
docs = leer_documentos(CARPETA_DOCS)



══════════════════════════════════════════════════════════
  NER — Extracción de entidades nombradas 23F
══════════════════════════════════════════════════════════

🧠 Cargando modelo de español (es_core_news_lg)...
   (si es la primera vez puede tardar unos segundos)

📂 Leyendo documentos...
  → 157 documentos leídos


In [42]:
docs[2]

('Capitán_Sánchez_Valiente_(2_de_junio_de_1981)',
 '88\n\nN.Rfa.: C/D13/10544/O2-06-81\n\nCalificación: C-2\n\nMICROFILMACION 10422/S\n\nNOTA INFORMATIVA\n\nAsunto: CAPITAN SANCHEZ VALIENTE.\n\nDurante los primeros meses de su permanencia en el extranjero el Capitán SANCHEZ VALIENTE empleó a su hermano, Capitán del Ejército del Aire, como "buzón" para el mantenimiento de los contactos con España. Sin embargo, a raíz del asalto al Banco Central de Barcelona, ha dejado de utilizarlo debido a que éste se encuentra muy vigilado y a que, al parecer, han surgido algunas desavenencias entre ambos al haberse negado el hermano a querellarse contra el Ministro de Defensa y el General de la IV Zona de la G.C. - a causa de las declaraciones de éstos en relación con la persona de SANCHEZ VALIENTE.\n\nActualmente parece que canaliza sus contactos a través de un antiguo Cabo de la Guardia Civil llamado JUAN ANTEQUERA BETRAN, - que fué uno de sus hombres de mayor confianza en el Grupo Operativo de la 

In [44]:
docs[0]

('Actitud_del_CESID_ante_la_situación_provocada_por_los_incidentes_en_el_Congreso',
 'ACTUACION DEL CESID ANTE LA SITUACION PROVOCADA POR LOS INCIDENTES EN EL CONGRESO\n\nCortada bruscamente por los acontecimientos la relación del CESID con su Mando natural, la Dirección en funciones del Centro, consciente de sus responsabilidades informativas, actuó en consecuencia.\n\nFundamentalmente, las relaciones se mantuvieron desde el primer momento en que se tuvo conocimiento de los hechos acaecidos con:\n\n- PREJUJEM (J-2) a base de dos tipos de actuaciones, unas de recomendaciones para:\n- que la Autoridad Militar asumiese el control y diese un comunicado por los m.c.s.;\n- que se realizase una acción psicológica sobre los ocupantes del Congreso por medios megatónicos;\n- que se aislase la zona del Congreso para cortar refuerzos;\n- acciones a llevar a cabo en caso de rendición; otras de tipo informativo:\n- situación de la zona de la capital y alrededores a base de las noticias recogidas de

In [45]:
   # 2. Limpiar y tokenizar
print("\n🧹 Limpiando texto...")
docs = limpiar_texto(docs)
print(f"  → {len(palabras):,} palabras tras limpiar")


🧹 Limpiando texto...
  → 157 palabras tras limpiar


In [46]:
docs[2]

('Capitán_Sánchez_Valiente_(2_de_junio_de_1981)',
 ['88',
  'rfa',
  'd13',
  '10544',
  'o2',
  '06',
  '81',
  'calificación',
  'microfilmacion',
  '10422',
  'nota',
  'informativa',
  'asunto',
  'capitan',
  'sanchez',
  'valiente',
  'primeros',
  'meses',
  'permanencia',
  'extranjero',
  'capitán',
  'sanchez',
  'valiente',
  'empleó',
  'hermano',
  'capitán',
  'ejército',
  'aire',
  'buzón',
  'mantenimiento',
  'contactos',
  'españa',
  'embargo',
  'raíz',
  'asalto',
  'banco',
  'central',
  'barcelona',
  'dejado',
  'utilizarlo',
  'debido',
  'éste',
  'encuentra',
  'vigilado',
  'parecer',
  'surgido',
  'desavenencias',
  'ambos',
  'haberse',
  'negado',
  'hermano',
  'querellarse',
  'ministro',
  'defensa',
  'general',
  'iv',
  'zona',
  'causa',
  'declaraciones',
  'éstos',
  'relación',
  'persona',
  'sanchez',
  'valiente',
  'actualmente',
  'parece',
  'canaliza',
  'contactos',
  'través',
  'antiguo',
  'cabo',
  'guardia',
  'civil',
  'llamado

In [ ]:
docs[1]

('Actitud_del_CESID_ante_la_situación_provocada_por_los_incidentes_en_el_Congreso',
 ['actuacion',
  'cesid',
  'situacion',
  'provocada',
  'incidentes',
  'congreso',
  'cortada',
  'bruscamente',
  'acontecimientos',
  'relación',
  'cesid',
  'mando',
  'natural',
  'dirección',
  'funciones',
  'centro',
  'consciente',
  'responsabilidades',
  'informativas',
  'actuó',
  'consecuencia',
  'fundamentalmente',
  'relaciones',
  'mantuvieron',
  'primer',
  'momento',
  'conocimiento',
  'hechos',
  'acaecidos',
  'prejujem',
  'base',
  'dos',
  'tipos',
  'actuaciones',
  'unas',
  'recomendaciones',
  'autoridad',
  'militar',
  'asumiese',
  'control',
  'diese',
  'comunicado',
  'realizase',
  'acción',
  'psicológica',
  'ocupantes',
  'congreso',
  'medios',
  'megatónicos',
  'aislase',
  'zona',
  'congreso',
  'cortar',
  'refuerzos',
  'acciones',
  'llevar',
  'cabo',
  'caso',
  'rendición',
  'tipo',
  'informativo',
  'situación',
  'zona',
  'capital',
  'alrededo